# 00 — Setup: Check Crankl and Build Shared Example Data

This first notebook does two simple jobs:

1. Make sure the real Crankl library and command-line tool are installed and working.
2. Create a small fixed set of example numbers that every later notebook will reuse.

**What is a crank word?**  
Crankl takes a list of floating-point weights and chops them into chunks of 64 numbers. Each chunk is treated as an 8×8 grid (like a tiny spreadsheet). That whole grid gets squeezed into one 64-bit integer called a **crank word** — think of it as a compressed fingerprint of the grid.

Our demo uses **two** such 8×8 grids as “source” data, plus slightly tweaked copies as “targets.” Later notebooks use that same data to show packing, updating, comparing, undoing steps, and running a forward pass.

## What each notebook covers

| Notebook | What it shows in plain terms |
|---|---|
| `01_clifford_crank` | Encode / decode numbers; basic algebra checks; turn a crank word back into an 8×8 grid |
| `02_sheaf_coboundary` | Pack grids into words; check how “connected” and how similar two packed archives look |
| `03_symplectic_turn` | Nudge crank words step by step (freely or toward a target); measure how much they changed |
| `04_rg_depth_peel` | Full file workflow: pack → update with history → roll back one step → compare files |
| `05_persistent_pack` | Pack and unpack; see how close the rebuilt grids are to the originals |
| `06_holonomy_forward` | Run packed weights on an input vector and look at the output |
| `07_parity_golden` | Confirm notebook math matches the official C++ tests |

> These notebooks call the real compiled C library (via `ctypes`) and the real CLI (via `subprocess`). They are demos of the actual product — not a separate NumPy rewrite.

In [ ]:
from pathlib import Path
import sys
import numpy as np

# Support execution from either the repository root or the notebooks directory.
notebook_dir = Path.cwd() / "notebooks" if (Path.cwd() / "notebooks").exists() else Path.cwd()
if str(notebook_dir) not in sys.path:
    sys.path.insert(0, str(notebook_dir))

from crankl_demo import CranklAPI, CLI, LIBRARY, prepare_demo_files, print_matrix, run_cli

np.set_printoptions(precision=3, suppress=True)
api = CranklAPI()
demo = prepare_demo_files()

print(f"Crankl C API version: {api.version()}")
print(f"Shared library:       {LIBRARY}")
print(f"CLI:                  {CLI}")
print(f"Temporary demo data:  {demo.source_path.parent}")
print(f"Source floats:        {demo.source_blocks.size}")
print(f"8x8 source blocks:    {len(demo.source_blocks)}")

## Look at the example matrices

We built two 8×8 source grids on purpose so they are easy to read when printed:

- **Block 0** — strongest values sit on the main diagonal; neighbors are weaker. Like a band around the diagonal.
- **Block 1** — a smooth pattern of positive/negative couplings, plus a diagonal boost.

Each **target** block is the same as its source, plus a tiny fixed sine-wave tweak. That gives later notebooks something nearby to “aim at.”

`prepare_demo_files()` also writes these numbers to temporary `.f32` files so the command-line tools can read them.

In [ ]:
for block_index, source_block in enumerate(demo.source_blocks):
    print_matrix(f"Source block {block_index}", source_block)
    print_matrix(f"Target block {block_index}", demo.target_blocks[block_index])
    print_matrix(
        f"Target perturbation {block_index}",
        demo.target_blocks[block_index] - source_block,
    )

## Confirm the command-line tool works

Crankl has two front doors to the same engine:

- the **CLI** — great for files and `.crank` archives
- the **C API** — great for working with words and matrices in Python memory

Later notebooks use both. Here we just print the version and list the demo files that were written.

In [ ]:
run_cli("version")

print("\nGenerated files:")
for path in (
    demo.source_path,
    demo.target_path,
    demo.calibration_x_path,
    demo.calibration_y_path,
):
    print(f"  {path.name:22s} {path.stat().st_size:4d} bytes")